In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/customer_transactions.csv')

display(df.head(10))

,date,store_nbr,lylty_card_nbr,txn_id,prod_nbr,prod_qty,tot_sales,clean_prod_name,net_weight,brand,lifestage,premium_customer
0,2018-10-17,1,1000,1,5,2,6.0,natural chip compny seasalt 175 g,175,Natural Chip Company,YOUNG SINGLES/COUPLES,Premium
1,2019-05-14,1,1307,348,66,3,6.3,ccs nacho cheese 175 g,175,CCs,MIDAGE SINGLES/COUPLES,Budget
2,2019-05-20,1,1343,383,61,2,2.9,smiths crinkle cut chips chicken 170 g,170,Smith's,MIDAGE SINGLES/COUPLES,Budget
3,2018-08-17,2,2373,974,69,5,15.0,smiths chip thinly s cream onion 175 g,175,Smith's,MIDAGE SINGLES/COUPLES,Budget
4,2018-08-18,2,2426,1038,108,3,13.8,kettle tortilla chpshny jlpno chili 150 g,150,Kettle,MIDAGE SINGLES/COUPLES,Budget
5,2019-05-16,4,4149,3333,16,1,5.7,smiths crinkle chips salt vinegar 330 g,330,Smith's,MIDAGE SINGLES/COUPLES,Budget
6,2019-05-16,4,4196,3539,24,1,3.6,grain waves sweet chilli 210 g,210,Grain Waves,MIDAGE SINGLES/COUPLES,Budget
7,2018-08-20,5,5026,4525,42,1,3.9,doritos corn chip mexican jalapeno 150 g,150,Doritos,MIDAGE SINGLES/COUPLES,Budget
8,2018-08-18,7,7150,6900,52,2,7.2,grain waves sour cream chives 210 g,210,Grain Waves,MIDAGE SINGLES/COUPLES,Budget
9,2019-05-17,7,7215,7176,16,1,5.7,smiths crinkle chips salt vinegar 330 g,330,Smith's,MIDAGE SINGLES/COUPLES,Budget


since pack size alone cannot help us narrow down each cohort's most preferred products, this analysis would utilize the combination of brand and pack size. <br>

- this is to acquire ample specificity, without flooding the results with the countless unique SKUs

In [9]:
# Create a Brand + Pack identifier before creating segment DataFrames.
df["brand_pack"] = (
    df["brand"] + " " + df["net_weight"].astype(int).astype(str) + "g"
)

# Naming: segment_<tier>_<lifestage> = target, comparison_<tier>_<lifestage> = its reference group.
# Each target gets its own comparison group, so nothing gets overwritten.

# budget older families
segment_b_olderfamilies = df[
    (df["lifestage"] == "OLDER FAMILIES") & (df["premium_customer"] == "Budget")
]
comparison_b_olderfamilies = df[
    df["lifestage"].isin(["OLDER SINGLES/COUPLES", "RETIREES"])
]

# budget young families
segment_b_youngfamilies = df[
    (df["lifestage"] == "YOUNG FAMILIES") & (df["premium_customer"] == "Budget")
]
comparison_b_youngfamilies = df[
    df["lifestage"].isin(["YOUNG SINGLES/COUPLES"])   # removed YOUNG FAMILIES: it contained the target
]

# mainstream young singles/couples
segment_main_youngsc = df[
    (df["lifestage"] == "YOUNG SINGLES/COUPLES") & (df["premium_customer"] == "Mainstream")
]
comparison_main_youngsc = df[
    df["lifestage"].isin(["YOUNG FAMILIES"])
]

# older singles/couples
segment_older_sc = df[df["lifestage"] == "OLDER SINGLES/COUPLES"]
comparison_older_sc = df[
    df["lifestage"].isin(["OLDER FAMILIES", "RETIREES"])
]

# mainstream retirees
segment_main_retirees = df[
    (df["lifestage"] == "RETIREES") & (df["premium_customer"] == "Mainstream")
]
comparison_main_retirees = df[
    df["lifestage"].isin(["OLDER SINGLES/COUPLES", "OLDER FAMILIES"])
]

# Safety check: no segment may share rows with its comparison group
for seg, comp in [
    (segment_b_olderfamilies,  comparison_b_olderfamilies),
    (segment_b_youngfamilies,  comparison_b_youngfamilies),
    (segment_main_youngsc,     comparison_main_youngsc),
    (segment_older_sc,         comparison_older_sc),
    (segment_main_retirees,    comparison_main_retirees),
]:
    assert seg.index.intersection(comp.index).empty, "segment overlaps its comparison group"

In [11]:
# Brand + Pack affinity (Mainstream Young Singles/Couples)
brand_pack_affinity = pd.merge(
    (
        segment_main_youngsc.groupby("brand_pack")["prod_qty"].sum()
        / segment_main_youngsc["prod_qty"].sum()
    ).rename("target_segment"),
    (
        comparison_main_youngsc.groupby("brand_pack")["prod_qty"].sum()
        / comparison_main_youngsc["prod_qty"].sum()
    ).rename("other"),
    on="brand_pack"
)

brand_pack_affinity["affinity_to_brand_pack"] = (
    brand_pack_affinity["target_segment"]
    / brand_pack_affinity["other"]
)

brand_pack_affinity = (
    brand_pack_affinity
    .sort_values("affinity_to_brand_pack", ascending=False)
    .reset_index()
)

display(brand_pack_affinity)

,brand_pack,target_segment,other,affinity_to_brand_pack
0,Twisties 270g,0.031943,0.023601,1.353453
1,Doritos 330g,0.015487,0.011455,1.352004
2,Smith's 380g,0.016512,0.012300,1.342390
3,Doritos 380g,0.015764,0.012006,1.313035
4,Doritos 170g,0.062945,0.048177,1.306544
5,Kettle 150g,0.092229,0.071035,1.298357
6,Infuzions 110g,0.061865,0.048125,1.285484
7,Tyrrells 165g,0.031666,0.024767,1.278551
8,Kettle 175g,0.091647,0.072316,1.267308
9,Smith's 330g,0.031002,0.024486,1.266115


In [13]:
import numpy as np
import pandas as pd

def classify_affinity(score):
    """Tiers are mirrored on the ratio scale (1.20 <-> 1/1.20), so over- and
    under-indexing are held to the same standard."""
    if pd.isna(score):
        return "No Data"
    if score >= 1.20:        return "Very High"      # 20%+ above
    elif score >= 1.10:      return "High"           # 10-20% above
    elif score >= 1.05:      return "Moderate High"  # 5-10% above
    elif score >= 1 / 1.05:  return "Neutral"        # roughly 0.95 to 1.05
    elif score >= 1 / 1.10:  return "Moderate Low"   # roughly 0.91 to 0.95
    elif score >= 1 / 1.20:  return "Low"            # roughly 0.83 to 0.91
    else:                    return "Very Low"       # below roughly 0.83


def affinity_table(segment, other, item_col="brand", value_col="prod_qty", z=1.96):
    """Affinity per item with a significance flag.

    significant = True when the score differs from 1 by more than sampling
    noise would explain (approximate 95% test on the log ratio). Rows are
    treated as independent, so this slightly overstates confidence because
    units from the same customer are correlated.
    """
    a = segment.groupby(item_col)[value_col].sum()
    b = other.groupby(item_col)[value_col].sum()
    A, B = a.sum(), b.sum()

    t = pd.DataFrame({"seg_units": a, "other_units": b}).fillna(0)
    with np.errstate(divide="ignore", invalid="ignore"):
        t["affinity"] = (t["seg_units"] / A) / (t["other_units"] / B)
        se = np.sqrt(1 / t["seg_units"] - 1 / A + 1 / t["other_units"] - 1 / B)
        t["significant"] = np.abs(np.log(t["affinity"])) > z * se
    t["tier"] = t["affinity"].map(classify_affinity)
    return t.sort_values("affinity", ascending=False)

brand_affinity = affinity_table(segment_main_youngsc, comparison_main_youngsc, item_col="brand")

display(brand_affinity)

,seg_units,other_units,affinity,significant,tier
brand,,,,,
Twisties,1673,2746,1.317343,True,Very High
Tyrrells,1143,1933,1.278551,True,Very High
Kettle,7172,12153,1.276029,True,Very High
Doritos,4447,7546,1.274248,True,Very High
Pringles,4326,7435,1.258083,True,Very High
Tostitos,1645,2837,1.253748,True,Very High
Cobs,1617,2932,1.192476,True,High
Infuzions,2343,4286,1.182017,True,High
Thins,2187,4218,1.121104,True,High
